# Home Credit – GCI – Detailed EDA

This is the exploratory data analysis (EDA) section of a credit-default prediction notebook (evaluation metric: ROC AUC). It is **self-contained**: it only reads `train.csv` and `test.csv` from the provided data, uses no external data, and uses no labels from test (the test set is only used for missing-value rates, train-vs-test comparisons and structure checks). It answers questions about the data and does not train a model to predict the label (a small LightGBM is used in a few places only to measure whether *one column can explain another column*; it is not used for any submission).

**Contents**
1. Overview and data quality (size, types, duplicates, label)
2. Missing values: how many, how they are missing together, whether missingness is related to the label
3. Unusual values (sentinels, spikes, `XNA` strings, logic violations)
4. **Testing hypotheses about missing and unusual values**
   - H1: is `days_employed = 365243` "not working"? (by income type, education, age, organization, occupation...)
   - H2: does a missing `own_car_age` mean "no car"?
   - H3: what is the structure of the four bureau columns that are missing together?
   - H4: `ext_source_1/2/3` missing: random or for a reason?
   - H5: `occupation_type` missing
   - H6: **all** the remaining columns with NaN, using the same testing framework
5. Per-column profile (meaning, hypothesis, statistics, distribution, transform, univariate) and 5b. Bivariate
6. Train vs test (is there a distribution shift?)
7. Correlation between variables
8. Near-duplicate records (data structure)
9. Summary of findings

**How to read the numbers.** Default rates always come with `n` and a 95% confidence interval (Wilson). Small groups have wide confidence intervals, so do not over-read them. The strength of the association between two columns ("strength 0–1") is Cramer's V for categorical columns and |2·AUC−1| for numeric columns; the two scales are only roughly comparable.

## 0. Setup and data loading

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.titleweight": "bold"})
SEED = 42
KAGGLE_PATH = "/kaggle/input/datasets/kiethe/homecredit-gci"
DATA_PATH = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else "."
ID_COL, TARGET = "customer_loan_id", "target"
RENAME_MAP = {"sk_id_curr": "customer_loan_id", "amt_income_total": "income", "amt_credit": "credit_amount", "amt_annuity": "annuity"}


def load(name):
    df = pd.read_csv(f"{DATA_PATH}/{name}.csv")
    df.columns = df.columns.str.lower()
    return df.rename(columns=RENAME_MAP)


train, test = load("train"), load("test")
full = pd.concat([train.assign(_set="train"), test.assign(_set="test")], ignore_index=True)
tr = full[full["_set"] == "train"].reset_index(drop=True)
te = full[full["_set"] == "test"].reset_index(drop=True)
y = tr[TARGET].astype(int).to_numpy()
BASE = y.mean()
FINDINGS = []     # (section, verdict, evidence): summary table in section 9


def is_cat(s):
    return s.dtype == object or pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype)


RAW = [c for c in train.columns if c not in (ID_COL, TARGET)]
CAT = [c for c in RAW if is_cat(train[c])]
NUM = [c for c in RAW if c not in CAT]
print(f"train {train.shape} | test {test.shape} | default rate {BASE:.2%} | {len(CAT)} categorical columns, {len(NUM)} numeric columns")


# ---------- shared helpers ----------
def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    nn = np.maximum(n, 1)
    p = k / nn
    den = 1 + z * z / nn
    c = (p + z * z / (2 * nn)) / den
    h = z * np.sqrt(p * (1 - p) / nn + z * z / (4 * nn ** 2)) / den
    return c - h, c + h


def rate_table(keys, tgt=None, min_n=1):
    # default-rate table by group; NaN is merged into a "(missing)" group
    k = pd.Series(keys).reset_index(drop=True)
    g = pd.DataFrame({"k": k.astype(object).where(k.notna(), "(missing)"), "y": y if tgt is None else tgt})
    t = g.groupby("k")["y"].agg(n="size", bad="sum")
    t["share"] = t["n"] / t["n"].sum()
    t["rate"] = t["bad"] / t["n"]
    t["ci_lo"], t["ci_hi"] = wilson(t["bad"], t["n"])
    t["lift"] = t["rate"] / BASE
    return t[t["n"] >= min_n].sort_values("rate", ascending=False)


def plot_rates(t, title, ax=None, top=14, sort_by_n=False):
    t = (t.sort_values("n", ascending=False).head(top) if sort_by_n else t.head(top)).iloc[::-1]
    ax = ax or plt.subplots(figsize=(7, 0.38 * len(t) + 1.2))[1]
    ax.barh([str(i)[:34] for i in t.index], t["rate"], color="#4c78a8", alpha=0.85)
    ax.errorbar(t["rate"], range(len(t)), xerr=[t["rate"] - t["ci_lo"], t["ci_hi"] - t["rate"]], fmt="none", ecolor="#222", lw=1)
    ax.axvline(BASE, color="#c44e52", ls="--", lw=1)
    for i, (r, n) in enumerate(zip(t["rate"], t["n"])):
        ax.text(r, i, f"  {r:.1%} (n={n:,})", va="center", fontsize=8)
    ax.set_title(title)
    ax.set_xlim(0, max(t["ci_hi"].max() * 1.35, BASE * 1.5))
    ax.set_xlabel("default rate (red line = average, error bars = 95% CI)")
    return ax


def cramers_v(a, b):
    ct = pd.crosstab(pd.Series(a).astype(object).where(pd.Series(a).notna(), "(missing)"), pd.Series(b).astype(object))
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return np.nan
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return float(np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))))


def assoc_table(mask, df, exclude=(), top=15):
    # how strongly each column is associated with the indicator `mask`: Cramer's V (categorical) or |2*AUC-1| (numeric)
    m = np.asarray(mask, bool)
    rows = []
    for c in df.columns:
        if c in exclude or c in (ID_COL, TARGET, "_set"):
            continue
        s = df[c]
        if is_cat(s) or s.nunique(dropna=False) <= 10:
            v = cramers_v(s.to_numpy(), m)
            if not np.isnan(v):
                rows.append((c, "categorical", v))
        else:
            x = pd.to_numeric(s, errors="coerce").to_numpy(float)
            ok = ~np.isnan(x)
            if ok.sum() >= 100 and 0 < m[ok].sum() < ok.sum():
                rows.append((c, "numeric", abs(roc_auc_score(m[ok], x[ok]) - 0.5) * 2))
            # whether column `c` is missing may also be associated with mask
            na = ~ok
            if 0 < na.sum() < len(na) and 0 < m[na].sum():
                rows.append((c + " [is missing?]", "missing", cramers_v(na, m)))
    t = pd.DataFrame(rows, columns=["column", "type", "strength (0-1)"]).sort_values("strength (0-1)", ascending=False)
    return t.head(top).reset_index(drop=True)


def predict_mask_auc(mask, df, exclude=(), n=80000, folds=3):
    # AUC of a small LightGBM predicting the indicator `mask` from the remaining columns (measures how "explainable" it is)
    cols = [c for c in df.columns if c not in exclude and c not in (ID_COL, TARGET, "_set")]
    X = df[cols].copy()
    for c in cols:
        if is_cat(X[c]):
            X[c] = X[c].astype("category")
    m = np.asarray(mask, bool).astype(int)
    rs = np.random.RandomState(SEED)
    idx = rs.choice(len(X), min(n, len(X)), replace=False)
    Xs, ms = X.iloc[idx].reset_index(drop=True), m[idx]
    if ms.sum() < 30 or ms.sum() > len(ms) - 30:
        return np.nan, pd.Series(dtype=float)
    oof = np.zeros(len(ms))
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(Xs, ms):
        mdl = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, verbose=-1, random_state=SEED)
        mdl.fit(Xs.iloc[a], ms[a])
        oof[b] = mdl.predict_proba(Xs.iloc[b])[:, 1]
    mdl.fit(Xs, ms)
    imp = pd.Series(mdl.booster_.feature_importance("gain"), index=cols).sort_values(ascending=False)
    return roc_auc_score(ms, oof), (imp / imp.sum()).head(6)


def note(section, verdict, evidence):
    FINDINGS.append({"section": section, "verdict": verdict, "evidence": evidence})
    print(f"[{section}] {verdict}\n    {evidence}")

## 1. Overview and data quality

In [ ]:
info = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "unique values": train.nunique(dropna=True),
    "missing train %": train.isna().mean() * 100,
    "missing test %": test.reindex(columns=train.columns).isna().mean() * 100,
    "examples": [", ".join(map(str, train[c].dropna().unique()[:3]))[:44] for c in train.columns],
})
display(info)
print(f"Train memory: {train.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Duplicate {ID_COL}: train {train[ID_COL].duplicated().sum()}, between train and test {len(set(train[ID_COL]) & set(test[ID_COL]))}")
cols_noid = [c for c in train.columns if c not in (ID_COL, TARGET)]
print(f"Fully duplicated rows (excluding ID and label): train {train.duplicated(subset=cols_noid).sum()}")
const = [c for c in train.columns if train[c].nunique(dropna=False) <= 1]
print("Constant columns:", const or "none")
print(f"Label rate: {BASE:.2%} ({y.sum():,} / {len(y):,}); class imbalance ratio {int((1 - BASE) / BASE)}:1")

## 2. Missing values

Three questions: (a) how much each column is missing, (b) in which patterns columns are missing together, (c) whether missingness goes together with the label. Remember that *missing* is only meaningful once we know **why** it is missing; section 4 tests each hypothesis.

In [ ]:
miss = pd.DataFrame({"train": train.isna().mean() * 100, "test": test.reindex(columns=train.columns).isna().mean() * 100})
miss = miss[(miss["train"] > 0) | (miss["test"] > 0)].sort_values("train", ascending=False)
display(miss)
ax = miss.plot.barh(figsize=(7, 0.45 * len(miss) + 1.2), color=["#4c78a8", "#f58518"])
ax.invert_yaxis(); ax.set_xlabel("% missing"); ax.set_title("Missing rate by column, train and test")
plt.show()

In [ ]:
na_cols = [c for c in miss.index if c != TARGET]
na_ind = tr[na_cols].isna()
# (b) joint missingness patterns
pat = na_ind.astype(int).astype(str).agg("".join, axis=1)
pt = pd.DataFrame({"pattern": pat, "y": y}).groupby("pattern")["y"].agg(n="size", rate="mean").sort_values("n", ascending=False)
pt["share"] = pt["n"] / len(tr)
pt["missing columns"] = [", ".join(c for c, b in zip(na_cols, p) if b == "1") or "(no column in the list is missing)" for p in pt.index]
display(pt.head(12)[["n", "share", "rate", "missing columns"]])
print("Column order in the pattern:", na_cols)

# phi coefficient between missingness indicators
phi = na_ind.astype(float).corr()
fig, ax = plt.subplots(figsize=(0.6 * len(na_cols) + 3, 0.6 * len(na_cols) + 2))
im = ax.imshow(phi, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(na_cols))); ax.set_xticklabels(na_cols, rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(len(na_cols))); ax.set_yticklabels(na_cols, fontsize=8)
ax.grid(False); ax.set_title("Phi coefficient between missingness indicators (1 = always missing together)")
for i in range(len(na_cols)):
    for j in range(len(na_cols)):
        ax.text(j, i, f"{phi.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, shrink=0.7); plt.show()

In [ ]:
# (c) is missingness related to the label: default rate when missing vs present, two-proportion z test
rows = []
for c in na_cols:
    m = tr[c].isna().to_numpy()
    k1, n1, k0, n0 = y[m].sum(), m.sum(), y[~m].sum(), (~m).sum()
    p1, p0, pp = k1 / n1, k0 / n0, (k1 + k0) / (n1 + n0)
    z = (p1 - p0) / np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n0))
    rows.append({"column": c, "n missing": n1, "default rate if missing": p1, "default rate if present": p0, "difference (pp)": 100 * (p1 - p0), "z": z, "p": 2 * stats.norm.sf(abs(z))})
mt = pd.DataFrame(rows).set_index("column").sort_values("z", key=np.abs, ascending=False)
display(mt)
note("2. missingness vs label", f"{int((mt['p'] < 0.01).sum())}/{len(mt)} columns have a significantly different default rate (p<0.01) between the missing and non-missing groups",
     "; ".join(f"{c}: {r['difference (pp)']:+.1f}pp" for c, r in mt.head(4).iterrows()))

## 3. Unusual values

Three kinds are scanned automatically: (a) **spikes**: a value repeated unusually often compared with its neighbouring values (a sign of a sentinel such as 365243), (b) **strings that look like "missing"** (`XNA`, `Unknown`...), (c) **logic violations** between columns (age, employment duration, household size...).

In [ ]:
MAGIC = {365243, 99999, 9999, 999, -1, -999, 999999, 1900, 65535, 9999999}
rows = []
for c in NUM:
    s = tr[c].dropna()
    if s.nunique() < 12:
        continue
    vc = s.value_counts()
    top_v, top_n = vc.index[0], vc.iloc[0]
    ratio = top_n / max(vc.iloc[1:21].mean(), 1)
    magic = [v for v in vc.head(10).index if v in MAGIC]
    if (ratio >= 5 and top_n >= 0.005 * len(s)) or magic:
        rows.append({"column": c, "most frequent value": top_v, "row count": top_n, "% train": 100 * top_n / len(tr), "times the mean count of the next 20 values": ratio,
                     "matches a known 'magic' value": magic or ""})
spike = pd.DataFrame(rows).set_index("column") if rows else pd.DataFrame()
print("Numeric columns with a spike or a magic value:")
display(spike if len(spike) else "none")

# strings that look like missing
sus = ["XNA", "Unknown", "unknown", "nan", "NaN", "None", "", "?", "N/A", "n/a"]
rows = []
for c in CAT:
    vc = tr[c].value_counts(dropna=False)
    for v in vc.index:
        if str(v) in sus:
            rows.append({"column": c, "value": v, "row count": vc[v], "% train": 100 * vc[v] / len(tr), "default rate": y[(tr[c] == v).to_numpy()].mean()})
print("Strings that look like 'missing' in categorical columns:")
display(pd.DataFrame(rows).set_index("column") if rows else "none")

In [ ]:
# logic violations: one rule per entry, count the violating rows (train and test)
d = full.copy()
d["age_years"] = -d["days_birth"] / 365.25
d["emp_years_raw"] = np.where(d["days_employed"] == 365243, np.nan, -d["days_employed"] / 365.25)
d["reg_years"] = -d["days_registration"] / 365.25
d["id_years"] = -d["days_id_publish"] / 365.25
rules = {
    "age < 18 or > 80": (d["age_years"] < 18) | (d["age_years"] > 80),
    "days_employed > 0 but ≠ 365243": (d["days_employed"] > 0) & (d["days_employed"] != 365243),
    "employment duration > age - 14": d["emp_years_raw"] > d["age_years"] - 14,
    "registration duration > age": d["reg_years"] > d["age_years"],
    "ID document issued before age 14": d["id_years"] > d["age_years"] - 14,
    "days_last_phone_change > 0": d["days_last_phone_change"] > 0,
    "family members < children + 1": d["cnt_fam_members"] < d["cnt_children"] + 1,
    "income ≤ 0": d["income"] <= 0,
    "income > 10 million": d["income"] > 1e7,
    "loan amount ≤ 0": d["credit_amount"] <= 0,
    "annuity ≤ 0": d["annuity"] <= 0,
    "goods price > 1.5 × loan amount": d["amt_goods_price"] > 1.5 * d["credit_amount"],
    "children ≥ 10": d["cnt_children"] >= 10,
    "ext_source outside [0,1]": ((d[["ext_source_1", "ext_source_2", "ext_source_3"]] < 0) | (d[["ext_source_1", "ext_source_2", "ext_source_3"]] > 1)).any(axis=1),
    "bureau: requests in 1 hour ≥ 5": d["amt_req_credit_bureau_hour"] >= 5,
    "bureau: requests in 3 months ≥ 20": d["amt_req_credit_bureau_qrt"] >= 20,
    "own_car_age ≥ 60": d["own_car_age"] >= 60,
}
rv = pd.DataFrame({"train": {k: int(v[d["_set"] == "train"].sum()) for k, v in rules.items()},
                   "test": {k: int(v[d["_set"] == "test"].sum()) for k, v in rules.items()}})
rv["% train"] = 100 * rv["train"] / len(tr)
rv["default rate when violated"] = [np.nan if r == 0 else y[rules[k][d["_set"] == "train"].to_numpy()].mean() for k, r in rv["train"].items()]
display(rv)

## 4. Testing hypotheses about missing and unusual values

Each hypothesis follows the same framework: **statement → measure the coverage and precision of the explanation → try alternative explanations (education, age, organization...) → conditional conclusion**. The original column is always kept as is; `365243` is not replaced by NaN before the testing.

### H1. Is `days_employed = 365243` "not working"?

In [ ]:
S = tr["days_employed"].eq(365243)
print(f"365243: {S.sum():,} rows ({S.mean():.2%} of train; test {te['days_employed'].eq(365243).mean():.2%}); default rate {y[S].mean():.2%} vs {y[~S].mean():.2%} in the rest")

# (1) by income type
g = tr.assign(S=S).groupby("name_income_type").agg(n=("S", "size"), has_365243=("S", "sum"))
g["% with 365243"] = 100 * g["has_365243"] / g["n"]
g["default rate"] = pd.Series(y).groupby(tr["name_income_type"]).mean()
display(g.sort_values("% with 365243", ascending=False))
core = list(g.index[g["% with 365243"] >= 90])
prec = float(S[tr["name_income_type"].isin(core)].mean()) if core else np.nan
rec = float(tr.loc[S, "name_income_type"].isin(core).mean()) if S.any() else np.nan
print(f"Rule 'income type ∈ {core}': precision (within that group, % with 365243) = {prec:.1%}, coverage (among the 365243 rows, % in that group) = {rec:.1%}")

In [ ]:
# (2) 365243 rows that are NOT in the core group, and the reverse: core group rows without 365243
out = tr[S & ~tr["name_income_type"].isin(core)]
inn = tr[~S & tr["name_income_type"].isin(core)]
print(f"365243 outside the core group: {len(out):,} rows | core group without 365243: {len(inn):,} rows")
if len(out):
    print("Income type of the rows outside the core:"); display(out["name_income_type"].value_counts().to_frame("n"))
if len(inn):
    print("Core group rows with a normal days_employed (examples):"); display(inn[["name_income_type", "days_employed", "days_birth", "organization_type"]].head(5))

# (3) compare the 365243 group with the others on 'life situation' columns
prof = pd.DataFrame({
    "365243": [S.sum(), -tr.loc[S, "days_birth"].median() / 365.25, tr.loc[S, "flag_emp_phone"].eq(0).mean() if "flag_emp_phone" in tr else np.nan,
               tr.loc[S, "organization_type"].eq("XNA").mean(), tr.loc[S, "occupation_type"].isna().mean(), tr.loc[S, "income"].median(), y[S].mean()],
    "rest": [(~S).sum(), -tr.loc[~S, "days_birth"].median() / 365.25, tr.loc[~S, "flag_emp_phone"].eq(0).mean() if "flag_emp_phone" in tr else np.nan,
                tr.loc[~S, "organization_type"].eq("XNA").mean(), tr.loc[~S, "occupation_type"].isna().mean(), tr.loc[~S, "income"].median(), y[~S].mean()],
}, index=["rows", "median age", "% flag_emp_phone = 0", "% organization_type = XNA", "% missing occupation_type", "median income", "default rate"])
display(prof)

In [ ]:
# (4) Do EDUCATION and the other columns explain anything more?
# If education is associated with 365243 only BECAUSE it is associated with income type, then within each income type education is no longer associated.
edu = tr["name_education_type"]
v_all = cramers_v(edu.to_numpy(), S.to_numpy())
inc = tr["name_income_type"]
rows = [{"scope": "all rows", "n": len(tr), "Cramer's V (education ~ has 365243)": v_all}]
for t_ in g.sort_values("n", ascending=False).index[:6]:
    m = (inc == t_).to_numpy()
    if S[m].nunique() > 1:
        rows.append({"scope": f"within '{t_}'", "n": int(m.sum()), "Cramer's V (education ~ has 365243)": cramers_v(edu[m].to_numpy(), S[m].to_numpy())})
    else:
        rows.append({"scope": f"within '{t_}'", "n": int(m.sum()), "Cramer's V (education ~ has 365243)": np.nan})
display(pd.DataFrame(rows))
ct = pd.crosstab(edu, S, normalize="index").rename(columns={True: "% with 365243", False: "% without"}) * 100
display(ct)

# education × income type together: share with 365243
display((tr.assign(S=S).pivot_table(index="name_education_type", columns="name_income_type", values="S", aggfunc="mean") * 100).round(1))

# (5) how strongly every column is associated with 365243 (including numeric columns such as age)
at = assoc_table(S.to_numpy(), tr, exclude=("days_employed",), top=12)
display(at)
auc_all, imp = predict_mask_auc(S.to_numpy(), tr, exclude=("days_employed",))
auc_inc, _ = predict_mask_auc(S.to_numpy(), tr[["name_income_type"]].assign(days_employed=tr["days_employed"]), exclude=("days_employed",))
print(f"AUC predicting '365243' from all other columns = {auc_all:.4f}; from income type only = {auc_inc:.4f}")
display(imp.to_frame("gain share"))

if not S.any():
    note("H1: 365243", "NO 365243 values in train", "there are no rows to test")
elif core and rec >= 0.95 and prec >= 0.90:
    note("H1: 365243", f"CONFIRMED: 365243 marks the non-working group ({', '.join(core)})",
         f"coverage {rec:.1%}, precision {prec:.1%}; V(education) overall {v_all:.2f}; AUC from income type {auc_inc:.3f}")
else:
    note("H1: 365243", "ONLY PARTLY TRUE or unclear", f"coverage {rec:.1%}, precision {prec:.1%} (thresholds 95%/90%); see the tables above")

**How to read H1.** If coverage and precision are both high, then 365243 really means "no job" (retired, unemployed...) and is not random, and replacing it with NaN and creating a "not working" flag is reasonable. If education is only associated when we do not split by income type (high overall V, low V within each group), then education explains nothing extra; it simply follows income type (older people tend to have lower education).

### H2. Does a missing `own_car_age` mean "no car"?

In [ ]:
car_na = tr["own_car_age"].isna()
print(f"own_car_age missing: {car_na.mean():.2%}. The table has NO flag_own_car column, so this cannot be tested directly; only indirect evidence is available.")
print("Car-related columns in the table:", [c for c in tr.columns if "car" in c or "own" in c])
obs = tr["own_car_age"].dropna()
vc = obs.value_counts().head(8)
print("Most common values of own_car_age:", dict(vc))
print(f"Car age distribution (when present): median {obs.median():.0f}, p95 {obs.quantile(.95):.0f}, max {obs.max():.0f}; ≥ 60 years: {(obs >= 60).mean():.2%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(obs.clip(upper=obs.quantile(.995)), bins=60, color="#4c78a8"); ax[0].set_title("own_car_age (when present), clipped at p99.5"); ax[0].set_xlabel("car age (years)")
rt = rate_table(pd.cut(obs, [-1, 0, 2, 5, 9, 14, 24, 200]).astype(str), tgt=y[~car_na.to_numpy()])
plot_rates(rt.sort_index(), "Default rate by car-age group", ax=ax[1], top=10)
plt.tight_layout(); plt.show()

# is missing consistent with "no car"? owning a car implies a certain income / circumstances
cmp = pd.DataFrame({
    "has own_car_age": [tr.loc[~car_na, "income"].median(), tr.loc[~car_na, "credit_amount"].median(), -tr.loc[~car_na, "days_birth"].median() / 365.25,
                       tr.loc[~car_na, "region_rating_client"].mean(), (~car_na).sum(), y[~car_na.to_numpy()].mean()],
    "missing": [tr.loc[car_na, "income"].median(), tr.loc[car_na, "credit_amount"].median(), -tr.loc[car_na, "days_birth"].median() / 365.25,
              tr.loc[car_na, "region_rating_client"].mean(), car_na.sum(), y[car_na.to_numpy()].mean()],
}, index=["median income", "median loan amount", "median age", "mean region rating", "n", "default rate"])
display(cmp)
display(assoc_table(car_na.to_numpy(), tr, exclude=("own_car_age",), top=10))
auc_c, imp_c = predict_mask_auc(car_na.to_numpy(), tr, exclude=("own_car_age",))
print(f"AUC predicting 'own_car_age missing' from all other columns = {auc_c:.3f} (0.5 = completely random, 1.0 = fully explained)")
note("H2: own_car_age", "CANNOT BE TESTED DIRECTLY; missing is usually read as 'no car'",
     f"missing {car_na.mean():.0%}; AUC predicting from other columns {auc_c:.2f}; default-rate difference missing vs present {100 * (y[car_na.to_numpy()].mean() - y[~car_na.to_numpy()].mean()):+.1f} pp")

### H3. The four bureau columns missing together

In [ ]:
BUR = [c for c in tr.columns if c.startswith("amt_req_credit_bureau")]
bm = tr[BUR].isna()
all_same = bool((bm.nunique(axis=1) == 1).all())
print(f"Columns: {BUR}")
print(f"Missing: {bm.mean().round(4).to_dict()} | all 4 always missing together: {all_same} (share of rows missing at least one: {bm.any(axis=1).mean():.2%})")
B = bm.any(axis=1)
display(assoc_table(B.to_numpy(), tr, exclude=BUR, top=10))
auc_b, imp_b = predict_mask_auc(B.to_numpy(), tr, exclude=BUR)
print(f"AUC predicting 'bureau missing' from all other columns = {auc_b:.3f}")
display(imp_b.to_frame("gain share"))

# missingness by group
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, c in zip(ax, ("name_income_type", "occupation_type", "region_rating_client")):
    gg = pd.DataFrame({"k": tr[c].astype(object).where(tr[c].notna(), "(missing)"), "m": B}).groupby("k")["m"].agg(["mean", "size"]).sort_values("mean")
    gg = gg[gg["size"] >= 50].tail(10)
    a.barh(gg.index.astype(str).str[:28], 100 * gg["mean"], color="#4c78a8"); a.set_title(f"% bureau missing by {c}", fontsize=9); a.set_xlabel("%")
plt.tight_layout(); plt.show()

# values when present: mostly zero? any outliers?
print("When present: quick statistics")
display(tr[BUR].describe(percentiles=[.5, .9, .99, .999]).T[["count", "mean", "50%", "90%", "99%", "99.9%", "max"]])
print("Share equal to 0:", (tr[BUR] == 0).sum().div(tr[BUR].notna().sum()).round(3).to_dict())
yb = y[B.to_numpy()].mean(); yn = y[~B.to_numpy()].mean()
note("H3: bureau", ("CONFIRMED: the 4 columns are missing together (a single flag)" if all_same else "the 4 columns are NOT always missing together"),
     f"missing {B.mean():.1%}; AUC predicting missingness from other columns {auc_b:.2f}; default rate {yb:.2%} (missing) vs {yn:.2%} (present)")

### H4. `ext_source_1/2/3` missing: random or for a reason?

In [ ]:
EXT = ["ext_source_1", "ext_source_2", "ext_source_3"]
rows = []
for c in EXT:
    m = tr[c].isna().to_numpy()
    others = [e for e in EXT if e != c]
    auc_all_, imp_ = predict_mask_auc(m, tr, exclude=(c,))
    auc_noext, _ = predict_mask_auc(m, tr, exclude=tuple(EXT))
    rows.append({"column": c, "% missing": 100 * m.mean(), "AUC from all other columns": auc_all_, "AUC without any ext_source columns": auc_noext,
                 "default rate if missing": y[m].mean() if m.any() else np.nan, "default rate if present": y[~m].mean(),
                 "top explaining columns": ", ".join(imp_.index[:3]) if len(imp_) else ""})
et = pd.DataFrame(rows).set_index("column")
display(et)
print("Missing together (row counts by missingness combination):")
display(tr[EXT].isna().astype(int).astype(str).agg("".join, axis=1).value_counts().rename_axis("pattern (1 = missing: ext1 ext2 ext3)").to_frame("n"))

# Hint of MNAR (missing not at random): distribution of ext_2 and ext_3 when ext_1 is missing vs present
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for a, (c, o) in zip(ax, (("ext_source_1", "ext_source_2"), ("ext_source_1", "ext_source_3"), ("ext_source_3", "ext_source_2"))):
    m = tr[c].isna()
    a.hist(tr.loc[~m, o].dropna(), bins=50, alpha=0.55, density=True, label=f"{c} present", color="#4c78a8")
    a.hist(tr.loc[m, o].dropna(), bins=50, alpha=0.55, density=True, label=f"{c} missing", color="#f58518")
    a.set_title(f"distribution of {o} by {c} missing/present", fontsize=9); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

# KS test between the two groups
for c, o in (("ext_source_1", "ext_source_2"), ("ext_source_1", "ext_source_3"), ("ext_source_3", "ext_source_2")):
    m = tr[c].isna()
    ks = stats.ks_2samp(tr.loc[~m, o].dropna(), tr.loc[m, o].dropna())
    print(f"KS of {o} between the {c}-missing and {c}-present groups: D = {ks.statistic:.3f}, p = {ks.pvalue:.2g}")
best = et["AUC from all other columns"].max()
note("H4: ext_source", ("STRUCTURED MISSINGNESS (explainable from other columns)" if best >= 0.7 else "MISSINGNESS NEARLY RANDOM relative to the other columns" if best < 0.6 else "MISSINGNESS MODERATELY ASSOCIATED"),
     "; ".join(f"{c}: missing {r['% missing']:.1f}%, AUC {r['AUC from all other columns']:.2f}" for c, r in et.iterrows()))

### H5. `occupation_type` missing

In [ ]:
occ_na = tr["occupation_type"].isna()
print(f"occupation_type missing: {occ_na.mean():.2%}; default rate when missing {y[occ_na.to_numpy()].mean():.2%} vs {y[~occ_na.to_numpy()].mean():.2%}")
g5 = tr.assign(m=occ_na).groupby("name_income_type")["m"].agg(["size", "mean"]).rename(columns={"size": "n", "mean": "% missing occupation"})
g5["% missing occupation"] *= 100
display(g5.sort_values("% missing occupation", ascending=False))
print(f"Overlap with 365243: {(occ_na & S).sum():,} rows are both missing occupation and 365243 ({(occ_na & S).sum() / max(occ_na.sum(), 1):.1%} of the rows missing occupation)")
print("Split by 365243:")
display(pd.crosstab(occ_na.rename("occupation missing"), S.rename("365243"), normalize="columns").mul(100).round(1))
display(assoc_table(occ_na.to_numpy(), tr, exclude=("occupation_type",), top=10))
auc_o, imp_o = predict_mask_auc(occ_na.to_numpy(), tr, exclude=("occupation_type",))
only_S = float((occ_na & ~S).sum() / max(occ_na.sum(), 1))
print(f"AUC predicting 'occupation missing' from all other columns = {auc_o:.3f}; share of missing occupation NOT in the 365243 group: {only_S:.1%}")
note("H5: occupation_type", "STRUCTURED MISSINGNESS" if auc_o >= 0.7 else "fairly random missingness", f"missing {occ_na.mean():.1%}; AUC from other columns {auc_o:.2f}; {only_S:.0%} of missing rows are outside the 365243 group")

### H6. Testing ALL columns that have NaN (not only the ones above)

The same framework for every column with missing values: (1) how well the other columns explain the missingness (AUC), (2) the best simple rule "column X = value v implies missing" (precision and coverage), (3) which column it is missing together with, (4) the default rate when missing. It also scans columns that are *not* missing but contain values that look like missing (sentinels, `XNA`).

In [ ]:
def best_rule(mask, df, exclude, min_n=30):
    # the rule 'column = v  =>  missing' with the highest F1
    m = np.asarray(mask, bool)
    best = (0, None, None, 0, 0)
    for c in df.columns:
        if c in exclude or c in (ID_COL, TARGET, "_set"):
            continue
        s_ = df[c]
        if s_.nunique(dropna=False) <= 1 or not (is_cat(s_) or s_.nunique(dropna=False) <= 30):
            continue
        k = s_.astype(object).where(s_.notna(), "(NaN)")
        g_ = pd.DataFrame({"k": k, "m": m}).groupby("k")["m"].agg(["sum", "size"])
        g_ = g_[g_["size"] >= min_n]
        if g_.empty or m.sum() == 0:
            continue
        prec_, rec_ = g_["sum"] / g_["size"], g_["sum"] / m.sum()
        f1 = 2 * prec_ * rec_ / (prec_ + rec_ + 1e-12)
        f1 = f1.where(prec_ >= 1.3 * m.mean(), 0)   # the rule's precision must exceed the base rate by at least 30%
        v = f1.idxmax()
        if f1[v] > best[0] and f1[v] > 0:
            best = (float(f1[v]), c, v, float(prec_[v]), float(rec_[v]))
    return best


rows = []
for c in [c for c in na_cols if c != TARGET]:
    m = tr[c].isna().to_numpy()
    auc_, imp_ = predict_mask_auc(m, tr, exclude=(c,))
    f1, rc, rv_, pr, rcl = best_rule(m, tr, exclude=(c,))
    co = na_ind.drop(columns=c).corrwith(pd.Series(m, index=na_ind.index).astype(float)).sort_values(key=np.abs, ascending=False)
    rows.append({"column": c, "% missing train": 100 * m.mean(), "% missing test": 100 * te[c].isna().mean(),
                 "AUC from other columns": auc_, "best single rule": f"{rc} = {rv_}" if rc else "", "precision": pr, "coverage": rcl,
                 "missing together with": (f"{co.index[0]} (phi {co.iloc[0]:.2f})" if len(co) and abs(co.iloc[0]) >= 0.3 else "—"),
                 "default rate (missing)": y[m].mean(), "default rate (present)": y[~m].mean()})
allna = pd.DataFrame(rows).set_index("column")
def _verdict(r):
    if r["AUC from other columns"] >= 0.9 or (r["precision"] >= 0.9 and r["coverage"] >= 0.9):
        return "CLEAR STRUCTURE (derivable from other columns)"
    if r["missing together with"] != "—":
        return "missing in a cluster with other columns"
    if r["AUC from other columns"] >= 0.65:
        return "moderate association"
    return "nearly random relative to the other columns"
allna["verdict"] = allna.apply(_verdict, axis=1)
display(allna.sort_values("% missing train", ascending=False))
for c, r in allna.iterrows():
    note(f"H6: {c}", r["verdict"], f"missing {r['% missing train']:.1f}%; AUC {r['AUC from other columns']:.2f}; rule {r['best single rule'] or '—'} (precision {r['precision']:.0%}, coverage {r['coverage']:.0%})")

# columns that are NOT missing but contain values that look like missing (sentinel / XNA): checked with the same framework
print()
sus_rows = []
for c in RAW:
    if c in na_cols:
        continue
    s_ = tr[c]
    cand = []
    if is_cat(s_):
        cand = [v for v in s_.dropna().unique() if str(v).strip() in {"XNA", "Unknown", "unknown", "", "?", "N/A"}]
    else:
        vc_ = s_.value_counts()
        cand = [v for v in vc_.head(5).index if v in MAGIC and vc_[v] >= 30]
    for v in cand:
        m = (s_ == v).to_numpy()
        auc_, _ = predict_mask_auc(m, tr, exclude=(c,))
        f1, rc, rv_, pr, rcl = best_rule(m, tr, exclude=(c,))
        sus_rows.append({"column": c, "value": v, "% train": 100 * m.mean(), "AUC from other columns": auc_, "best single rule": f"{rc} = {rv_}" if rc else "",
                         "precision": pr, "coverage": rcl, "default rate": y[m].mean()})
if sus_rows:
    sus_t = pd.DataFrame(sus_rows).set_index("column")
    display(sus_t)
    for c, r in sus_t.iterrows():
        note(f"H6: {c}={r['value']}", "value that looks like missing: " + ("CLEAR STRUCTURE" if r["AUC from other columns"] >= 0.9 else "not yet explained"), f"{r['% train']:.1f}% of rows; AUC {r['AUC from other columns']:.2f}; rule {r['best single rule']}")
else:
    print("No column outside the NaN list has a value that looks like missing.")

## 5. Per-column profile: meaning, hypothesis, statistics, distribution, transform, relationship with the label

Each column gets a "card" with: **(1) business meaning**, **(2) hypothesis** (the expected direction of the effect on default, taken from business knowledge, not from the data), **(3) descriptive statistics** (quantiles, skewness, kurtosis, % outliers, % zeros...), **(4) distribution** (by label), **(5) whether to transform** (skewness before/after log, sqrt, Yeo-Johnson) and **(6) relationship with the label** (Mann-Whitney / χ² tests, shape of the default-rate curve by quantile, and the conclusion on whether the hypothesis holds).

Important conventions:
- The `days_*` columns are converted to **years** (positive) for readability; `days_employed = 365243` is set apart (tested in section 4) and excluded from the statistics of `years_employed`.
- Decision trees are **invariant to monotonic transforms**, so the "transform" recommendation only matters for linear models, neural networks (TabM) and ratio features.
- The "hypothesis" is a business expectation; the conclusion at the bottom of the card is what the data says.

In [ ]:
# ---------------- data dictionary: meaning + hypothesis ----------------
# direction: +1 increase -> more defaults, -1 increase -> fewer, 0 no clear expectation
DICT = {
    "name_contract_type": ("Contract type: cash loan (Cash) or card / revolving credit line (Revolving).", "Cash loans are riskier than Revolving (large loans, fixed installments).", ["Cash loans"]),
    "code_gender": ("Borrower's gender (XNA = unknown).", "Men default more than women; XNA is too rare to conclude anything.", ["M"]),
    "cnt_children": ("Number of children.", "More children -> heavier spending burden -> more defaults.", +1),
    "income": ("Total annual income.", "Higher income -> fewer defaults but a weak effect; heavily right-skewed distribution (needs log).", -1),
    "credit_amount": ("Loan amount.", "Direction unclear: large loans go with high income but also with a large debt burden.", 0),
    "annuity": ("Installment amount per period.", "High installment -> cash-flow pressure -> more defaults.", +1),
    "amt_goods_price": ("Price of the goods the loan is used to buy.", "Almost identical to the loan amount (credit ≈ goods); little separate information.", 0),
    "name_income_type": ("Source of income (Working, Pensioner, Commercial associate...).", "Pensioner/State servant are low risk; Unemployed, Maternity leave are the riskiest.", ["Unemployed", "Maternity leave", "Working"]),
    "name_education_type": ("Education level.", "Lower education -> more defaults (natural order: Lower secondary > Secondary > Incomplete higher > Higher > Academic).", ["Lower secondary", "Secondary / secondary special"]),
    "name_family_status": ("Marital status.", "Single / civil marriage are riskier than married.", ["Single / not married", "Civil marriage", "Separated"]),
    "name_housing_type": ("Type of housing.", "Living with parents or renting is riskier than owning a home.", ["With parents", "Rented apartment"]),
    "region_population_relative": ("Relative population density of the region (normalized).", "Densely populated (urban) regions have fewer defaults.", -1),
    "age_years": ("Age (years), from -days_birth/365.25.", "Younger people default more, monotonically decreasing with age.", -1),
    "years_employed": ("Years in the current job (365243 = not working, excluded).", "Longer employment -> stability -> fewer defaults.", -1),
    "years_registration": ("Years since the last change of residence registration.", "Recent registration change (few years) -> instability -> more defaults.", -1),
    "years_id_publish": ("Years since the last change of identity document.", "Recent document change (few years) -> more defaults.", -1),
    "own_car_age": ("Age of the borrower's car (NaN = no car, according to hypothesis H2).", "Direction unclear; a very new car may indicate a better-off person.", 0),
    "flag_mobil": ("Provided a mobile phone number (constant column).", "Constant, carries no information.", 0),
    "flag_emp_phone": ("Has a work phone number.", "Having a work number = currently employed -> lower risk.", -1),
    "flag_work_phone": ("Has a workplace phone number (separate).", "Direction unclear.", 0),
    "occupation_type": ("Occupation (NaN = not working, according to H5).", "Manual labor / drivers are riskier than office staff / managers.", ["Laborers", "Drivers", "Low-skill Laborers", "Sales staff", "Cooking staff"]),
    "cnt_fam_members": ("Number of household members.", "Larger household -> heavier burden -> more defaults (reinforces cnt_children).", +1),
    "region_rating_client": ("Rating of the region where the borrower lives (1 best, 3 worst).", "The worse the rating (larger number), the more defaults.", +1),
    "organization_type": ("Type of organization where the borrower works (many levels).", "Some industries (construction, transport, self-employed) are riskier; XNA = not working.", ["Transport: type 3", "Construction", "Self-employed"]),
    "ext_source_1": ("Credit score from external source 1 (0-1, high = good).", "High score -> fewer defaults (the strongest signal).", -1),
    "ext_source_2": ("Credit score from external source 2.", "High score -> fewer defaults.", -1),
    "ext_source_3": ("Credit score from external source 3.", "High score -> fewer defaults.", -1),
    "years_since_phone_change": ("Years since the last phone number change.", "Recent number change -> instability -> more defaults.", -1),
    "amt_req_credit_bureau_hour": ("Number of credit bureau queries in the 1 hour before the application.", "A burst of queries -> applying in many places -> more defaults (very rare).", +1),
    "amt_req_credit_bureau_mon": ("Number of credit bureau queries in 1 month.", "Many -> riskier.", +1),
    "amt_req_credit_bureau_qrt": ("Number of credit bureau queries in 1 quarter.", "Many -> riskier.", +1),
    "amt_req_credit_bureau_year": ("Number of credit bureau queries in 1 year.", "Many -> riskier.", +1),
}
VIEW = {   # days_* column -> (new name, function)
    "days_birth": ("age_years", lambda s: -s / 365.25),
    "days_employed": ("years_employed", lambda s: -s.where(s != 365243) / 365.25),
    "days_registration": ("years_registration", lambda s: -s / 365.25),
    "days_id_publish": ("years_id_publish", lambda s: -s / 365.25),
    "days_last_phone_change": ("years_since_phone_change", lambda s: -s / 365.25),
}
V = tr.copy()
for src, (nm, fn) in VIEW.items():
    if src in V.columns:
        V[nm] = fn(V[src])
NUM_V = [VIEW[c][0] if c in VIEW else c for c in NUM]
NUM_V = [c for c in NUM_V if c in V.columns and V[c].nunique() > 1]
CAT_V = [c for c in CAT if V[c].nunique() > 1]
PROFILE = []


def hist_pair(ax, s, title, clip=True):
    for lab, col in ((0, "#4c78a8"), (1, "#c44e52")):
        v = s[y == lab].dropna()
        if clip and len(v) > 50:
            v = v.clip(v.quantile(0.005), v.quantile(0.995))
        ax.hist(v, bins=45, alpha=0.5, density=True, color=col, label=f"label {lab}")
    ax.set_title(title, fontsize=9); ax.tick_params(labelsize=7)


def bin_rates(s):
    # group by value (discrete column) or decile; NaN becomes its own group. Returns a table (n, bad, rate, ci) in value order
    if s.nunique(dropna=True) <= 12:
        key = s
    else:
        key = pd.qcut(s.rank(method="first"), 10, labels=False).where(s.notna())
        key = key.map(lambda q: np.nan if pd.isna(q) else q)
    g_ = pd.DataFrame({"k": key, "y": y}).dropna(subset=["k"]).groupby("k")["y"].agg(n="size", bad="sum")
    g_["rate"] = g_["bad"] / g_["n"]
    g_["lo"], g_["hi"] = wilson(g_["bad"], g_["n"])
    return g_


def shape_of(rate):
    r = np.asarray(rate, float)
    if len(r) < 4:
        return "few groups"
    rho = stats.spearmanr(np.arange(len(r)), r)[0]
    if rho >= 0.85:
        return "monotonic increasing"
    if rho <= -0.85:
        return "monotonic decreasing"
    am, an = int(np.argmax(r)), int(np.argmin(r))
    if 0 < am < len(r) - 1 and (r[am] - r[[0, -1]].max()) > 0.2 * (r.max() - r.min()):
        return "bell-shaped ∩ (peak in the middle)"
    if 0 < an < len(r) - 1 and (r[[0, -1]].min() - r[an]) > 0.2 * (r.max() - r.min()):
        return "U-shaped (minimum in the middle)"
    return "non-monotonic / noisy"


def suggest_transform(x):
    # returns (name, transformed array, description, skew table)
    from sklearn.preprocessing import PowerTransformer
    x = x.dropna().to_numpy(float)
    sk = {"original": stats.skew(x)}
    cand = {}
    if x.min() >= 0:
        cand["log1p"] = np.log1p(x); cand["sqrt"] = np.sqrt(x)
    else:
        cand["signed log"] = np.sign(x) * np.log1p(np.abs(x))
    try:
        cand["Yeo-Johnson"] = PowerTransformer(method="yeo-johnson").fit_transform(x.reshape(-1, 1)).ravel()
    except Exception:
        pass
    for k, v in cand.items():
        sk[k] = stats.skew(v)
    if abs(sk["original"]) < 0.5:
        return "none needed", x, "already roughly symmetric; only z-score standardization is needed for linear models/NN", sk
    name = min(cand, key=lambda k: abs(sk[k]))
    simple = [k for k in ("log1p", "signed log") if k in cand]
    if simple and abs(sk[simple[0]]) <= abs(sk[name]) + 0.25 and abs(sk[simple[0]]) < 0.75:
        name = simple[0]
    return name, cand[name], f"skewness {sk['original']:+.2f} -> {sk[name]:+.2f} after {name}", sk


def num_card(c):
    s = V[c]
    meaning, hyp, exp = DICT.get(c, ("(no description yet)", "(no hypothesis yet)", 0))
    x = s.dropna()
    q = x.quantile([.01, .05, .25, .5, .75, .95, .99])
    iqr = q[.75] - q[.25]
    st = pd.Series({
        "n non-missing": len(x), "% missing": 100 * s.isna().mean(), "unique values": x.nunique(), "mean": x.mean(), "std dev": x.std(),
        "min": x.min(), "p1": q[.01], "p5": q[.05], "p25": q[.25], "median": q[.5], "p75": q[.75], "p95": q[.95], "p99": q[.99], "max": x.max(),
        "skewness": stats.skew(x), "kurtosis": stats.kurtosis(x), "% zero": 100 * (x == 0).mean(),
        "% Tukey outliers (1.5×IQR)": 100 * ((x < q[.25] - 1.5 * iqr) | (x > q[.75] + 1.5 * iqr)).mean() if iqr > 0 else 0.0,
        "% extreme outliers (3×IQR)": 100 * ((x < q[.25] - 3 * iqr) | (x > q[.75] + 3 * iqr)).mean() if iqr > 0 else 0.0,
    })
    # relationship with the label
    ok = s.notna().to_numpy()
    yy = y[ok]
    n1, n0 = yy.sum(), (1 - yy).sum()
    auc_ = roc_auc_score(yy, s[ok]) if n1 and n0 else np.nan
    z = (auc_ - 0.5) / np.sqrt((n1 + n0 + 1) / (12 * n1 * n0)) if n1 and n0 else np.nan
    pval = 2 * stats.norm.sf(abs(z))
    dirn = "increase -> more defaults" if auc_ > 0.5 else "increase -> fewer defaults"
    if exp == 0:
        verdict = "the hypothesis has no expected direction"
    elif pval > 0.01:
        verdict = "NO EVIDENCE of a direction (p > 0.01)"
    elif (auc_ > 0.5) == (exp > 0):
        verdict = "CORRECT: direction matches the hypothesis"
    else:
        verdict = "OPPOSITE: direction contradicts the hypothesis"
    br = bin_rates(s)
    sh = shape_of(br["rate"])
    ct = pd.crosstab(pd.qcut(s.rank(method="first"), 10, labels=False) if x.nunique() > 12 else s, y)
    chi_p = stats.chi2_contingency(ct)[1] if ct.shape[0] > 1 else np.nan
    # transform
    tname, xt, tdesc, sk = suggest_transform(s)
    pear_raw = np.corrcoef(x.to_numpy(float), yy)[0, 1]
    pear_tr = np.corrcoef(xt, yy)[0, 1] if tname != "none needed" else pear_raw
    # missing
    miss_line = ""
    if s.isna().any():
        m = s.isna().to_numpy()
        miss_line = f"When missing: default rate {y[m].mean():.2%} (n={m.sum():,}) vs {y[~m].mean():.2%} when present."
    if c == "years_employed":
        m365 = V["days_employed"].eq(365243).to_numpy()
        miss_line += f" Group 365243 (not working): default rate {y[m365].mean():.2%} (n={m365.sum():,})."

    print("=" * 110)
    print(f"{c}   |   univariate AUC {max(auc_, 1 - auc_):.3f} ({dirn})   |   {verdict}   |   shape: {sh}")
    print(f"Meaning   : {meaning}")
    print(f"Hypothesis: {hyp}")
    display(st.to_frame().T)
    print(f"Mann-Whitney: AUC={auc_:.4f}, Cliff's δ={2 * auc_ - 1:+.3f}, p={pval:.2g} | χ² by decile p={chi_p:.2g} | median label0={x[yy == 0].median():.3g}, label1={x[yy == 1].median():.3g}")
    print(f"Transform : {tname}: {tdesc} | Pearson correlation with label: original {pear_raw:+.4f}, after transform {pear_tr:+.4f} | skewness of the options: " + ", ".join(f"{k} {v:+.2f}" for k, v in sk.items()))
    if miss_line:
        print(miss_line)
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.3))
    hist_pair(ax[0], s, f"{c} (clipped p0.5-p99.5)"); ax[0].legend(fontsize=7)
    xs = pd.Series(xt)
    hist_pair(ax[1], pd.Series(xt, index=x.index).reindex(s.index), f"after transform: {tname}")
    stats.probplot((xs - xs.mean()) / (xs.std() + 1e-12), dist="norm", plot=ax[2]); ax[2].set_title("QQ plot vs normal (after transform)", fontsize=9); ax[2].tick_params(labelsize=7)
    xx = np.arange(len(br)); ax[3].errorbar(xx, br["rate"], yerr=[br["rate"] - br["lo"], br["hi"] - br["rate"]], fmt="-o", color="#4c78a8", ms=4, lw=1)
    ax[3].axhline(BASE, color="#c44e52", ls="--", lw=0.8)
    ax[3].set_xticks(xx); ax[3].set_xticklabels([f"{v:g}" for v in br.index], fontsize=7)
    ax[3].set_title("default rate by " + ("value" if x.nunique() <= 12 else "decile (1 = lowest)"), fontsize=9)
    plt.tight_layout(); plt.show()
    PROFILE.append({"column": c, "type": "numeric", "% missing": 100 * s.isna().mean(), "univariate AUC": max(auc_, 1 - auc_), "direction": dirn, "hypothesis": verdict, "shape": sh,
                    "skew": sk["original"], "recommended transform": tname, "meaning": meaning})


for _c in sorted(NUM_V, key=lambda c: -abs(roc_auc_score(y[V[c].notna().to_numpy()], V[c].dropna()) - 0.5) if V[c].notna().sum() > 200 and y[V[c].notna().to_numpy()].std() > 0 else 0):
    num_card(_c)

In [ ]:
# ---------------- categorical columns ----------------
ORDER = {"name_education_type": ["Lower secondary", "Secondary / secondary special", "Incomplete higher", "Higher education", "Academic degree"]}


def cat_card(c):
    s = V[c]
    meaning, hyp, exp_levels = DICT.get(c, ("(no description yet)", "(no hypothesis yet)", []))
    t = rate_table(s, min_n=1)
    k = t["bad"].sum()
    # per-level z vs the rest, Bonferroni
    nn, bb = t["n"].to_numpy(float), t["bad"].to_numpy(float)
    rest_p = (bb.sum() - bb) / np.maximum(nn.sum() - nn, 1)
    z = (t["rate"].to_numpy() - rest_p) / np.sqrt(np.maximum(rest_p * (1 - rest_p), 1e-12) * (1 / nn + 1 / np.maximum(nn.sum() - nn, 1)))
    t["z vs the rest"] = z
    t["significant (Bonferroni)"] = np.abs(z) > stats.norm.isf(0.025 / len(t))
    t["rare (<1%)"] = t["share"] < 0.01
    ct = pd.crosstab(s.astype(object).where(s.notna(), "(missing)"), y)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = float(np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))) if min(ct.shape) > 1 else np.nan
    top = list(t.index[:3])
    if exp_levels:
        hit = [lv for lv in exp_levels if lv in list(t.index[:max(3, len(exp_levels))])]
        verdict = (f"CORRECT (PARTLY OR FULLY): the levels expected to be risky are among the highest: {hit}" if hit else f"MISMATCH: the highest-risk levels are {top}, which do not match the expectation {exp_levels}")
    else:
        verdict = "no hypothesis about the levels"
    # encoding recommendation
    nlev = s.nunique(dropna=True)
    rare = int(t["rare (<1%)"].sum())
    if c in ORDER:
        enc = "natural order (ordinal); the ranking is already clear"
    elif nlev <= 7:
        enc = "one-hot or native categorical; few levels so it is safe"
    elif nlev <= 15:
        enc = "native categorical (LightGBM/CatBoost); consider merging rare levels"
    else:
        enc = "many levels: target encoding inside CV (prevents leakage) or CatBoost categorical; merge rare levels into 'Other'"
    if rare:
        enc += f"; {rare} rare levels (<1%) should be merged"
    print("=" * 110)
    print(f"{c}   |   {nlev} levels, {s.isna().mean():.1%} missing   |   Cramer's V={v:.3f}, χ² p={p:.2g}   |   {verdict}")
    print(f"Meaning   : {meaning}")
    print(f"Hypothesis: {hyp}")
    print(f"Recommended encoding: {enc}")
    display(t[["n", "share", "rate", "ci_lo", "ci_hi", "lift", "z vs the rest", "significant (Bonferroni)", "rare (<1%)"]].head(20))
    fig, ax = plt.subplots(1, 2, figsize=(14, 0.32 * min(len(t), 14) + 1.8), gridspec_kw={"width_ratios": [1.3, 1]})
    plot_rates(t, f"default rate by {c}", ax=ax[0], top=14)
    sh_ = t.sort_values("n", ascending=False).head(14).iloc[::-1]
    ax[1].barh([str(i)[:30] for i in sh_.index], 100 * sh_["share"], color="#999"); ax[1].set_title("% of rows", fontsize=9); ax[1].tick_params(labelsize=7)
    plt.tight_layout(); plt.show()
    PROFILE.append({"column": c, "type": "categorical", "% missing": 100 * s.isna().mean(), "univariate AUC": np.nan, "direction": f"V={v:.3f}", "hypothesis": verdict,
                    "shape": f"{nlev} levels", "skew": np.nan, "recommended transform": enc, "meaning": meaning})


_vs = {c: cramers_v(V[c].to_numpy(), y) for c in CAT_V}
for _c in sorted(CAT_V, key=lambda c: -np.nan_to_num(_vs[c])):
    cat_card(_c)

### 5.x Per-column summary table (meaning, hypothesis verdict, transform recommendation)

In [ ]:
prof = pd.DataFrame(PROFILE).set_index("column")
prof = prof.sort_values("univariate AUC", ascending=False, na_position="last")
display(prof[["type", "% missing", "univariate AUC", "direction", "hypothesis", "shape", "skew", "recommended transform"]])
ok_h = prof["hypothesis"].str.startswith("CORRECT").sum(); bad_h = prof["hypothesis"].str.startswith(("OPPOSITE", "MISMATCH", "NO EVIDENCE")).sum()
note("5. hypotheses per column", f"{ok_h} columns match the hypothesis, {bad_h} columns opposite/mismatched/no evidence, out of {len(prof)} columns",
     "; ".join(f"{c}: {r['hypothesis'][:26]}" for c, r in prof[prof['hypothesis'].str.startswith(('OPPOSITE', 'MISMATCH'))].head(5).iterrows()) or "no column goes against its hypothesis")

## 5b. Bivariate relationships

Covers: (1) whether **ratios / derived variables** are worth creating, (2) relationships between pairs of numeric variables, (3) **default-rate heatmaps** for important pairs, (4) an **interaction** scan (does a two-variable model beat each variable alone?), (5) categorical × categorical, (6) categorical × numeric.

In [ ]:
def cv_auc(cols, n=40000, folds=3, trees=120, df=None, return_oof=False):
    df = V if df is None else df
    X = df[cols].copy()
    for c in cols:
        if is_cat(X[c]):
            X[c] = X[c].astype("category")
    rs = np.random.RandomState(SEED); idx = rs.choice(len(X), min(n, len(X)), replace=False)
    Xs, ys = X.iloc[idx].reset_index(drop=True), y[idx]
    oof = np.zeros(len(ys))
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(Xs, ys):
        m_ = lgb.LGBMClassifier(n_estimators=trees, learning_rate=0.06, num_leaves=8, max_depth=3, min_child_samples=100, subsample=0.8, subsample_freq=1,
                                colsample_bytree=1.0, verbose=-1, random_state=SEED)
        m_.fit(Xs.iloc[a], ys[a]); oof[b] = m_.predict_proba(Xs.iloc[b])[:, 1]
    return (roc_auc_score(ys, oof), oof, ys) if return_oof else roc_auc_score(ys, oof)


# derived variables
V["credit_to_income"] = V["credit_amount"] / V["income"].replace(0, np.nan)
V["annuity_to_income"] = V["annuity"] / V["income"].replace(0, np.nan)
V["credit_term"] = V["credit_amount"] / V["annuity"].replace(0, np.nan)
V["goods_to_credit"] = V["amt_goods_price"] / V["credit_amount"].replace(0, np.nan)
V["employed_to_age"] = V["years_employed"] / V["age_years"]
V["income_per_member"] = V["income"] / V["cnt_fam_members"].replace(0, np.nan)
V["ext_mean"] = V[["ext_source_1", "ext_source_2", "ext_source_3"]].mean(axis=1)
V["ext_min"] = V[["ext_source_1", "ext_source_2", "ext_source_3"]].min(axis=1)
V["ext_prod"] = V[["ext_source_1", "ext_source_2", "ext_source_3"]].prod(axis=1, min_count=2)
DER = {
    "credit_to_income": ["credit_amount", "income"], "annuity_to_income": ["annuity", "income"], "credit_term": ["credit_amount", "annuity"],
    "goods_to_credit": ["amt_goods_price", "credit_amount"], "employed_to_age": ["years_employed", "age_years"], "income_per_member": ["income", "cnt_fam_members"],
    "ext_mean": ["ext_source_1", "ext_source_2", "ext_source_3"], "ext_min": ["ext_source_1", "ext_source_2", "ext_source_3"], "ext_prod": ["ext_source_1", "ext_source_2", "ext_source_3"],
}
rows = []
for d_, parts in DER.items():
    parts = [p_ for p_ in parts if p_ in V.columns]
    if len(parts) < 2 or V[d_].notna().sum() < 500:
        continue
    def uni(c):
        ok = V[c].notna().to_numpy()
        a = roc_auc_score(y[ok], V[c][ok]); return max(a, 1 - a)
    a_d, a_p = uni(d_), max(uni(p_) for p_ in parts)
    c_parts = cv_auc(parts); c_both = cv_auc(parts + [d_])
    rows.append({"derived variable": d_, "built from": " , ".join(parts), "univariate AUC (derived)": a_d, "best univariate AUC of the components": a_p,
                 "CV AUC components only": c_parts, "CV AUC components + derived": c_both, "added gain": c_both - c_parts})
dd = pd.DataFrame(rows).set_index("derived variable").sort_values("added gain", ascending=False)
display(dd)
print("How to read: 'added gain' > 0.001 means the tree cannot build this ratio from its components by itself (worth adding as a feature); ≈ 0 means it is redundant.")
note("5b. derived variables", "worth creating: " + (", ".join(dd.index[dd["added gain"] > 0.001]) or "none"), "; ".join(f"{c}: {r['added gain']:+.4f}" for c, r in dd.head(4).iterrows()))

In [ ]:
# relationships between important pairs of numeric variables (hexbin + Spearman)
pairs = [("income", "credit_amount"), ("credit_amount", "amt_goods_price"), ("credit_amount", "annuity"), ("age_years", "years_employed"),
         ("age_years", "ext_mean"), ("ext_source_2", "ext_source_3"), ("ext_source_1", "ext_source_2"), ("age_years", "years_registration"),
         ("cnt_children", "cnt_fam_members"), ("income", "ext_mean")]
pairs = [(a, b) for a, b in pairs if a in V.columns and b in V.columns]
fig, axes = plt.subplots(2, int(np.ceil(len(pairs) / 2)), figsize=(4.2 * int(np.ceil(len(pairs) / 2)), 7.4)); axes = axes.ravel()
for a_, (p1, p2) in zip(axes, pairs):
    d_ = V[[p1, p2]].dropna()
    rho = stats.spearmanr(d_[p1], d_[p2])[0]
    xx, yy_ = d_[p1], d_[p2]
    lx = p1 in ("income", "credit_amount", "amt_goods_price", "annuity"); ly = p2 in ("income", "credit_amount", "amt_goods_price", "annuity")
    a_.hexbin(np.log10(xx.clip(lower=1)) if lx else xx, np.log10(yy_.clip(lower=1)) if ly else yy_, gridsize=35, cmap="Blues", mincnt=1, bins="log")
    a_.set_xlabel(("log10 " if lx else "") + p1, fontsize=8); a_.set_ylabel(("log10 " if ly else "") + p2, fontsize=8); a_.tick_params(labelsize=7)
    a_.set_title(f"Spearman ρ = {rho:+.2f}", fontsize=9)
for a_ in axes[len(pairs):]:
    a_.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# default-rate heatmaps for important pairs (numeric x numeric via quantiles, categorical x categorical)
def qbin(s, q=5):
    s = s.copy()
    if s.nunique() <= q:
        return s.astype(object).where(s.notna(), "(missing)")
    b = pd.qcut(s.rank(method="first"), q, labels=[f"Q{i + 1}" for i in range(q)])
    return b.astype(object).where(s.notna(), "(missing)")


def heat(a, b, q=5, min_n=60, ax=None):
    A = qbin(V[a], q) if not is_cat(V[a]) else V[a].astype(object).where(V[a].notna(), "(missing)")
    B = qbin(V[b], q) if not is_cat(V[b]) else V[b].astype(object).where(V[b].notna(), "(missing)")
    d_ = pd.DataFrame({"a": A, "b": B, "y": y})
    r = d_.pivot_table(index="a", columns="b", values="y", aggfunc="mean")
    n = d_.pivot_table(index="a", columns="b", values="y", aggfunc="size")
    r = r.where(n >= min_n)
    if ax is None:
        ax = plt.subplots(figsize=(6.3, 4.5))[1]
    im = ax.imshow(r.values.astype(float), cmap="YlOrRd", aspect="auto", vmin=0, vmax=min(0.35, np.nanmax(r.values) if np.isfinite(np.nanmax(r.values)) else 0.3))
    ax.set_xticks(range(r.shape[1])); ax.set_xticklabels([str(c)[:14] for c in r.columns], rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(r.shape[0])); ax.set_yticklabels([str(i)[:22] for i in r.index], fontsize=7)
    for i in range(r.shape[0]):
        for j in range(r.shape[1]):
            if np.isfinite(r.values[i, j]):
                ax.text(j, i, f"{100 * r.values[i, j]:.1f}", ha="center", va="center", fontsize=7)
    ax.grid(False); ax.set_xlabel(b + (" (Q1 = lowest)" if not is_cat(V[b]) else ""), fontsize=8); ax.set_ylabel(a + (" (Q1 = lowest)" if not is_cat(V[a]) else ""), fontsize=8)
    ax.set_title(f"% default: {a} × {b}  (cells with n<{min_n} left blank)", fontsize=9)
    return r


HP = [("age_years", "ext_mean"), ("ext_source_2", "ext_source_3"), ("name_education_type", "name_income_type"), ("region_rating_client", "income"),
      ("credit_term", "ext_mean"), ("name_contract_type", "code_gender"), ("age_years", "years_employed"), ("annuity_to_income", "ext_mean")]
HP = [(a, b) for a, b in HP if a in V.columns and b in V.columns]
fig, axes = plt.subplots(int(np.ceil(len(HP) / 2)), 2, figsize=(13, 4.6 * int(np.ceil(len(HP) / 2)))); axes = axes.ravel()
for a_, (p1, p2) in zip(axes, HP):
    heat(p1, p2, ax=a_)
for a_ in axes[len(HP):]:
    a_.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# interaction scan: does the pair (A, B) predict better than A and B separately?
cand = []
for c in NUM_V + CAT_V:
    if c in V.columns and V[c].nunique() > 1 and c not in ("flag_mobil",):
        cand.append(c)
single, oofs = {}, {}
for c in cand:
    try:
        a_, o_, ys_ = cv_auc([c], n=30000, folds=3, trees=80, return_oof=True)
        single[c], oofs[c] = a_, o_
    except Exception:
        pass
top = sorted(single, key=single.get, reverse=True)[:10]
print("Top 10 variables by single-variable CV AUC:", {c: round(single[c], 4) for c in top})
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
rows = []
from itertools import combinations
for a, b in combinations(top, 2):
    both = cv_auc([a, b], n=30000, folds=3, trees=80)
    additive = roc_auc_score(ys_, lg(oofs[a]) + lg(oofs[b]))     # sum of the two single-variable models, NO cross-interaction
    rows.append({"A": a, "B": b, "pair AUC (depth-3 trees)": both, "AUC of the two single models summed": additive, "TRUE INTERACTION = pair − sum": both - additive})
it = pd.DataFrame(rows).sort_values("TRUE INTERACTION = pair − sum", ascending=False)
display(it.head(12))
print("How to read: last column > 0.005 means this pair has signal that the two variables added together do not have (a true interaction). Near 0 means purely additive. Since the sample is the same small 40k, treat ±0.003 as noise.")

In [ ]:
# categorical x categorical: Cramer's V
cv_ = pd.DataFrame(index=CAT_V, columns=CAT_V, dtype=float)
for a in CAT_V:
    for b in CAT_V:
        cv_.loc[a, b] = 1.0 if a == b else cramers_v(V[a].to_numpy(), V[b].to_numpy())
fig, ax = plt.subplots(figsize=(0.7 * len(CAT_V) + 3, 0.7 * len(CAT_V) + 2.4))
im = ax.imshow(cv_.values.astype(float), cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(len(CAT_V))); ax.set_xticklabels(CAT_V, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(len(CAT_V))); ax.set_yticklabels(CAT_V, fontsize=8)
for i in range(len(CAT_V)):
    for j in range(len(CAT_V)):
        ax.text(j, i, f"{cv_.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, shrink=0.7); ax.set_title("Cramer's V between categorical columns"); plt.show()

# categorical x numeric: eta^2 (share of the numeric variable's variance explained by the groups), Kruskal-Wallis
keyn = [c for c in ("age_years", "years_employed", "income", "credit_amount", "credit_term", "ext_mean", "annuity_to_income") if c in V.columns]
rows = []
for c in CAT_V:
    for nn_ in keyn:
        d_ = V[[c, nn_]].dropna()
        grp = [g_[nn_].to_numpy() for _, g_ in d_.groupby(c) if len(g_) >= 20]
        if len(grp) < 2:
            continue
        H, p = stats.kruskal(*grp)
        eta = (H - len(grp) + 1) / (len(d_) - len(grp))
        rows.append({"categorical column": c, "numeric column": nn_, "η² (Kruskal)": max(eta, 0), "p": p})
eta = pd.DataFrame(rows).pivot(index="categorical column", columns="numeric column", values="η² (Kruskal)")
display(eta.style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))
print("How to read: high η² means the categorical groups explain a lot of the variation of the numeric column (for example name_income_type ~ years_employed).")

## 6. Train vs test

Two checks: (a) the **PSI** of each column (PSI < 0.1: stable; 0.1–0.25: mild shift; > 0.25: clear shift) and (b) **adversarial validation**: train a small LightGBM to tell train rows from test rows; AUC ≈ 0.5 means they cannot be told apart (no shift).

In [ ]:
def psi_num(a, b, bins=10):
    qs = np.unique(np.nanquantile(a, np.linspace(0, 1, bins + 1)))
    if len(qs) < 3:
        return np.nan
    qs[0], qs[-1] = -np.inf, np.inf
    pa = np.histogram(a[~np.isnan(a)], qs)[0] / max((~np.isnan(a)).sum(), 1)
    pb = np.histogram(b[~np.isnan(b)], qs)[0] / max((~np.isnan(b)).sum(), 1)
    # add a "missing" bin
    pa = np.append(pa * (1 - np.isnan(a).mean()), np.isnan(a).mean()); pb = np.append(pb * (1 - np.isnan(b).mean()), np.isnan(b).mean())
    pa, pb = np.clip(pa, 1e-4, None), np.clip(pb, 1e-4, None)
    return float(np.sum((pb - pa) * np.log(pb / pa)))


def psi_cat(a, b):
    ca = a.astype(object).where(a.notna(), "(missing)").value_counts(normalize=True)
    cb = b.astype(object).where(b.notna(), "(missing)").value_counts(normalize=True)
    idx = ca.index.union(cb.index)
    pa, pb = ca.reindex(idx).fillna(0).clip(1e-4), cb.reindex(idx).fillna(0).clip(1e-4)
    return float(np.sum((pb - pa) * np.log(pb / pa)))


rows = []
for c in RAW:
    if c in CAT:
        rows.append((c, psi_cat(tr[c], te[c])))
    else:
        rows.append((c, psi_num(pd.to_numeric(tr[c]).to_numpy(float), pd.to_numeric(te[c]).to_numpy(float))))
psi = pd.DataFrame(rows, columns=["column", "PSI"]).set_index("column").sort_values("PSI", ascending=False)
display(psi.head(12))

adv = full.drop(columns=[TARGET, ID_COL]).copy()
lab = (adv["_set"] == "test").astype(int).to_numpy()
adv = adv.drop(columns="_set")
auc_adv, imp_adv = predict_mask_auc(lab, adv.assign(_set="x").assign(_set2=0), exclude=("_set2",), n=100000)
print(f"Adversarial validation: AUC = {auc_adv:.4f}  (0.5 = train cannot be distinguished from test)")
display(imp_adv.to_frame("gain share"))
note("6. train vs test", "NO SHIFT" if auc_adv < 0.55 else "MILD SHIFT" if auc_adv < 0.65 else "CLEAR SHIFT",
     f"adversarial AUC {auc_adv:.3f}; largest PSI {psi['PSI'].iloc[0]:.3f} ({psi.index[0]})")

## 7. Correlation between variables

In [ ]:
d2 = tr.copy()
d2["age_years"] = -d2["days_birth"] / 365.25
d2["years_employed"] = np.where(d2["days_employed"] == 365243, np.nan, -d2["days_employed"] / 365.25)
d2["years_registration"] = -d2["days_registration"] / 365.25
d2["years_id_publish"] = -d2["days_id_publish"] / 365.25
d2["years_last_phone_change"] = -d2["days_last_phone_change"] / 365.25
d2["credit_term"] = d2["credit_amount"] / d2["annuity"].replace(0, np.nan)
d2["ext_mean"] = d2[EXT].mean(axis=1)
d2["target"] = y
ncols = [c for c in NUM if not c.startswith("days_") and c not in ("flag_mobil",)] + ["age_years", "years_employed", "years_registration", "years_id_publish", "years_last_phone_change", "credit_term", "ext_mean", "target"]
ncols = [c for c in dict.fromkeys(ncols) if c in d2.columns and d2[c].nunique() > 1]
cm = d2[ncols].sample(min(60000, len(d2)), random_state=SEED).corr(method="spearman")
fig, ax = plt.subplots(figsize=(0.45 * len(ncols) + 4, 0.45 * len(ncols) + 3))
im = ax.imshow(cm, vmin=-1, vmax=1, cmap="RdBu_r"); ax.grid(False)
ax.set_xticks(range(len(ncols))); ax.set_xticklabels(ncols, rotation=80, ha="right", fontsize=7)
ax.set_yticks(range(len(ncols))); ax.set_yticklabels(ncols, fontsize=7)
plt.colorbar(im, shrink=0.7); ax.set_title("Spearman rank correlation"); plt.show()

pairs = cm.where(np.triu(np.ones(cm.shape, bool), 1)).stack().rename("rho").reset_index()
pairs.columns = ["variable A", "variable B", "rho"]
print("Most strongly correlated pairs (|rho|):")
display(pairs.reindex(pairs["rho"].abs().sort_values(ascending=False).index).head(15))
print("Rank correlation with the label:")
display(cm["target"].drop("target").to_frame("rho with label").sort_values("rho with label", key=np.abs, ascending=False).head(12))

## 8. Near-duplicate records (data structure)

Some time-related columns (`days_birth`, `days_registration`, `days_id_publish`...) are fairly stable for each person. If the same set of values appears in several rows, it may be the same person applying for several loans. This section only describes that structure. *Exploiting the labels of duplicate records for prediction exploits a property of the dataset rather than learning credit behaviour, so the modelling notebook does not use it for its main model.*

In [ ]:
keys = [c for c in ("days_birth", "days_registration", "days_id_publish", "region_population_relative") if c in full.columns]
gid = full[keys].astype(str).agg("|".join, axis=1)
sz = gid.map(gid.value_counts())
print(f"Group key: {keys}")
print(f"Number of groups: {gid.nunique():,} | rows in groups with ≥ 2 rows: {(sz >= 2).mean():.2%} (train+test)")
print("Group size distribution:"); display(sz.value_counts().sort_index().head(8).rename_axis("size").to_frame("rows"))
# within labeled groups: do two rows in the same group share a label more often than chance?
lab_full = full[TARGET]
pairs_same, pairs_tot, pos_in = 0, 0, 0
grp = pd.DataFrame({"g": gid, "t": lab_full})
grp = grp[grp["t"].notna() & (sz >= 2)]
for _, v in grp.groupby("g")["t"]:
    v = v.to_numpy()
    if len(v) >= 2:
        k, n_ = v.sum(), len(v)
        pairs_tot += n_ * (n_ - 1) / 2
        pairs_same += k * (k - 1) / 2 + (n_ - k) * (n_ - k - 1) / 2
exp_same = BASE ** 2 + (1 - BASE) ** 2
if pairs_tot:
    print(f"Same-group pairs with the same label: {pairs_same / pairs_tot:.2%} vs {exp_same:.2%} if independent → "
          f"{'labels are correlated within groups' if pairs_same / pairs_tot > exp_same + 0.01 else 'nearly independent'}")
note("8. duplicate records", f"{(sz >= 2).mean():.1%} of rows are in groups of ≥ 2 rows", f"key {', '.join(keys)}")

## 9. Summary of findings

In [ ]:
print("Per-column profile:")
display(pd.DataFrame(PROFILE).set_index("column")[["type", "univariate AUC", "hypothesis", "shape", "recommended transform"]])
print("Key findings:")
display(pd.DataFrame(FINDINGS).set_index("section"))

**Suggestions for using the results in the model** (depending on the real conclusions in the tables above):
- If H1 holds: keep `365243 → NaN` and the "not working" flag (already present), and do not create extra ratio features that divide by `years_employed` for this group.
- If H3/H4 show structured missingness (high explanation AUC): the missing flag may carry information; if missingness is nearly random, ML imputation will help little.
- If section 6 shows an adversarial AUC close to 0.5: CV is trustworthy for the public leaderboard; if it is high, look at which columns shift (importance table) before optimizing further.